# Model Comparison

This notebook compares the trained classical models and DNN using the saved outputs from Phase 2.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_curve, auc

sns.set_theme(style='whitegrid', palette='Set2')

PROJECT_ROOT = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from preprocess import split_preprocess_smote

RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models' / 'saved'

## Load Metrics

In [ ]:
classical = pd.read_csv(RESULTS_DIR / 'classical_model_metrics.csv')
dnn = pd.read_csv(RESULTS_DIR / 'dnn_metrics.csv')

metric_cols = ['model', 'accuracy', 'precision', 'recall', 'f1', 'auc_roc']
leaderboard = pd.concat([classical[metric_cols], dnn[metric_cols]], ignore_index=True)
leaderboard = leaderboard.sort_values(['auc_roc', 'f1'], ascending=False).reset_index(drop=True)
leaderboard.insert(0, 'rank', range(1, len(leaderboard) + 1))
display(leaderboard)

## AUC-ROC And F1 Bar Chart

In [ ]:
plot_df = leaderboard.melt(
    id_vars=['model'],
    value_vars=['auc_roc', 'f1'],
    var_name='metric',
    value_name='score',
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x='model', y='score', hue='metric')
plt.ylim(0, 1)
plt.title('Model Comparison: AUC-ROC and F1')
plt.xlabel('Model')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## ROC Curves

In [ ]:
processed = split_preprocess_smote(dataset='heart', use_smote=True)
plt.figure(figsize=(8, 6))

for model_name in classical['model']:
    artifact = joblib.load(MODELS_DIR / f'{model_name}.pkl')
    model = artifact['model']
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(processed.X_test)[:, 1]
    else:
        y_score = model.decision_function(processed.X_test)
    fpr, tpr, _ = roc_curve(processed.y_test, y_score)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc(fpr, tpr):.3f})')

try:
    import tensorflow as tf
    dnn_model = tf.keras.models.load_model(MODELS_DIR / 'dnn.keras')
    y_score = dnn_model.predict(processed.X_test, verbose=0).ravel()
    fpr, tpr, _ = roc_curve(processed.y_test, y_score)
    plt.plot(fpr, tpr, label=f'dnn (AUC={auc(fpr, tpr):.3f})')
except Exception as exc:
    print(f'DNN ROC skipped: {exc}')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('ROC Curves On Held-Out Test Set')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Ranked Leaderboard

The table below ranks models by AUC-ROC first and F1 second. For this run, SVM has the strongest held-out AUC, while the DNN is competitive on F1.

In [ ]:
display(leaderboard.style.format({
    'accuracy': '{:.3f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1': '{:.3f}',
    'auc_roc': '{:.3f}',
}))